# Teks + tabel, tingkat lanjut: tumpuk OOF model teks + fitur tetangga → GBDT
**Kapan dipakai:** tabel dengan kolom teks dan kolom angka/kategori, setelah `teks_tabular`
jalan. Bedanya: teks tidak hanya dipadatkan (SVD), tetapi **diringkas jadi tebakan**:

1. **OOF model teks linear** (LogReg, Naive Bayes) di TF-IDF kata + huruf → peluang tiap kelas.
   Pola umum juara lomba teks+tabel Kaggle (mis. Avito Demand): fitur "ridge OOF".
2. **Tetangga terdekat** (kNN, kosinus): porsi label tetangga, kemiripan tetangga terdekat per
   kelas, kemiripan tertinggi (kembar/hampir kembar = lookup).
3. **Kemiripan ke pusat kelas** (centroid) di ruang TF-IDF.

Semua dihitung **di dalam lipatan** (baris validasi tidak pernah melihat labelnya sendiri),
lalu LightGBM menggabungkannya dengan kolom tabel. Ablasi menunjukkan sumbangan tiap kelompok.

**Data demo:** ulasan baju (sama dengan `teks_tabular`), target `rating` 1–5: F1 makro OOF
0,468 (test 0,482) vs 0,399 di `teks_tabular`. Data teks saja: kosongkan `CAT_COLS` dan
`DROP_COLS` (SmSA: 0,885 vs LogReg 0,859). Kalau sudah ada fine-tune kuat dan datanya teks saja,
`ensemble_oof` biasanya lebih baik (SmSA: GBDT di atas OOF IndoBERT 0,928 vs IndoBERT 0,930).

In [ ]:
import copy
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from scipy.sparse import hstack
from sentence_transformers import SentenceTransformer
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import ComplementNB
from sklearn.neighbors import NearestNeighbors

## Konfigurasi
Yang diganti saat lomba hanya sel ini (baris `# GANTI saat lomba`). Kolom angka dikenali
otomatis: semua kolom selain id, target, teks, kategori, dan kolom yang dibuang.

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/stacking_teks_tabular"
OOF_FOLDER = "../../../outputs/text/oof"

DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"   # GANTI saat lomba: sample dari panitia
KEY_CSV = f"{DATA_FOLDER}/baju_wanita_test_labels.csv"       # GANTI saat lomba (kunci, kalau ada)

ID_COL = "id"                                                # GANTI saat lomba
TARGET = "rating"                                            # GANTI saat lomba
SUB_COL = "rating"                                           # GANTI saat lomba: kolom label sample
TEXT_COLS = ["title", "review_text"]                         # GANTI saat lomba (boleh lebih dari 1)
CAT_COLS = ["clothing_id", "division_name", "department_name", "class_name"]   # GANTI saat lomba
DROP_COLS = ["recommended_ind"]                              # GANTI saat lomba: hasil cek bocor

MODEL_NAME = "lgbm_stacking_teks"   # nama file OOF: oof_<MODEL_NAME>.npy, test_<MODEL_NAME>.npy
N_FOLDS = 5
SEED = 42
N_NEIGHBORS = 20            # tetangga per baris (kNN)
EXTRA_OOF = []              # model lain di OOF_FOLDER sebagai fitur (lipatan harus sama: seed 42)
# EXTRA_OOF = ["ft_indobert"]   # OOF finetune_oof (+ VERBAL_COLS) = biasanya fitur terkuat
EMBEDDING_MODEL = "../../../pretrained/minilm_en"   # hanya dipakai alt vektor kalimat (sel kNN)
# EMBEDDING_MODEL = "../../../pretrained/minilm"      # multibahasa
# EMBEDDING_MODEL = "../../../pretrained/e5_base"     # multibahasa kuat, TEXT_PREFIX "query: "
TEXT_PREFIX = ""

## Baca data
Teks kosong → `""`, kategori kosong → `"kosong"`; angka kosong dibiarkan (LightGBM mengerti
NaN). Kolom teks digabung jadi satu teks (`full_train`, `full_test`) untuk model teks dan kNN.

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
for col in TEXT_COLS:
    train[col] = train[col].fillna("").astype(str)
    test[col] = test[col].fillna("").astype(str)
for col in CAT_COLS:
    train[col] = train[col].fillna("kosong").astype(str)
    test[col] = test[col].fillna("kosong").astype(str)
full_train = train[TEXT_COLS].agg(" . ".join, axis=1)
full_test = test[TEXT_COLS].agg(" . ".join, axis=1)

skip = [ID_COL, TARGET] + TEXT_COLS + CAT_COLS + DROP_COLS
NUM_COLS = []
for col in train.columns:
    if col not in skip:
        NUM_COLS.append(col)
CLASSES = np.unique(train[TARGET])
y_idx = np.searchsorted(CLASSES, train[TARGET].values)   # kelas → 0..K-1
print("train:", train.shape, "| test:", test.shape, "| kelas:", CLASSES)
print("kolom angka:", NUM_COLS)

## Buat sample_submission demo (HAPUS saat lomba)

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUB_COL] = CLASSES[0]
demo_sample.to_csv(SAMPLE_CSV, index=False)
print("sample demo ditulis:", SAMPLE_CSV, demo_sample.shape)

## Lipatan + matriks TF-IDF
Satu daftar lipatan (`folds`) dipakai semua tahap: model teks, kNN, dan LightGBM. Dengan
lipatan yang sama, baris validasi LightGBM tidak pernah ikut melatih fitur miliknya sendiri.
TF-IDF kata (1–2 gram) + huruf (`char_wb` 2–5) dilatih di train + test (tanpa label, tidak
bocor). Huruf menangkap salah ketik, singkatan, dan bahasa gaul.

In [ ]:
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(splitter.split(np.zeros(len(y_idx)), y_idx))

all_text = pd.concat([full_train, full_test], ignore_index=True)
word_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200000, sublinear_tf=True)
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3,
                             max_features=200000, sublinear_tf=True)
matrix = hstack([word_tfidf.fit_transform(all_text), char_tfidf.fit_transform(all_text)])
matrix = matrix.tocsr()
matrix_train = matrix[: len(train)]
matrix_test = matrix[len(train):]
print("TF-IDF:", matrix_train.shape, "| kelas terkecil:", np.bincount(y_idx).min())

## Fitur 1: OOF model teks linear
Tiap model dilatih per lipatan; baris validasi diberi peluang dari model yang tidak melihatnya
(OOF), test diberi rata-rata model per lipatan. Skor tiap model teks dicetak: kalau salah satu
sudah mendekati skor akhir, kolom tabel kurang berguna. Tambah model lain cukup satu baris.

In [ ]:
LEVEL1 = {"logreg": LogisticRegression(C=4.0, max_iter=2000),
          "nb": ComplementNB(alpha=0.3)}
# LEVEL1["svm"] = CalibratedClassifierCV(LinearSVC(C=0.3), cv=3)   # alt: impor dulu


def oof_probability(model, X, X_test):
    oof = np.zeros((X.shape[0], len(CLASSES)), dtype="float32")
    test_pred = np.zeros((X_test.shape[0], len(CLASSES)), dtype="float32")
    for idx_fit, idx_val in folds:
        fitted = clone(model).fit(X[idx_fit], y_idx[idx_fit])
        oof[idx_val] = fitted.predict_proba(X[idx_val])
        test_pred += fitted.predict_proba(X_test) / N_FOLDS
    return oof, test_pred


START = time.time()
level1_train = []
level1_test = []
level1_names = []
for name, model in LEVEL1.items():
    oof, test_pred = oof_probability(model, matrix_train, matrix_test)
    score = f1_score(y_idx, oof.argmax(axis=1), average="macro")
    print(f"{name}: F1 makro OOF {score:.4f} | detik {time.time() - START:.0f}")
    level1_train.append(oof)
    level1_test.append(test_pred)
    for label in CLASSES:
        level1_names.append(f"{name}_p{label}")

### OOF dari notebook lain (`EXTRA_OOF`)
OOF `finetune_oof`, `klasifikasi_embedding`, dll. ikut jadi fitur. Syarat: dibuat dari train
yang sama dengan `StratifiedKFold(5, shuffle, 42)`, sehingga baris validasinya sama dengan
`folds` di sini (kalau beda, skor CV jadi terlalu optimis).

In [ ]:
for name in EXTRA_OOF:
    extra = np.load(f"{OOF_FOLDER}/oof_{name}.npy")
    score = f1_score(y_idx, extra.argmax(axis=1), average="macro")
    print(f"{name}: {extra.shape} | F1 makro OOF {score:.4f}")
    level1_train.append(extra)
    level1_test.append(np.load(f"{OOF_FOLDER}/test_{name}.npy"))
    for label in CLASSES:
        level1_names.append(f"{name}_p{label}")
print("fitur OOF teks:", len(level1_names))

## Fitur 2: tetangga terdekat + pusat kelas
Per baris: kemiripan tetangga ke-1 (≈ 1 berarti kembar → labelnya hampir pasti sama, kalau
data tidak berisik), rata-rata kemiripan, porsi label tetangga berbobot kemiripan, kemiripan
tetangga terdekat tiap kelas, dan kemiripan ke pusat (rata-rata TF-IDF) tiap kelas.
Alternatif ruang: vektor kalimat pretrained (baris ber-`#`; GPU di PC panitia: `device="cuda"`).

In [ ]:
def centroid_similarity(ref, ref_y, query):
    columns = []
    for class_id in range(len(CLASSES)):
        center = np.asarray(ref[ref_y == class_id].mean(axis=0)).ravel()
        center = center / (np.linalg.norm(center) + 1e-9)
        columns.append(np.asarray(query @ center).ravel())
    return np.column_stack(columns)


def neighbor_block(ref, ref_y, query):
    finder = NearestNeighbors(n_neighbors=N_NEIGHBORS, metric="cosine").fit(ref)
    distance, index = finder.kneighbors(query)
    similarity = 1 - distance
    labels = ref_y[index]
    weight_sum = similarity.sum(axis=1) + 1e-9
    columns = [similarity[:, 0], similarity.mean(axis=1)]
    for class_id in range(len(CLASSES)):
        in_class = similarity * (labels == class_id)
        columns.append(in_class.sum(axis=1) / weight_sum)
        columns.append(in_class.max(axis=1))
    return np.column_stack(columns + [centroid_similarity(ref, ref_y, query)])


rows = folds[0][1]   # 20% train berstrata: semua kelas ada
print("uji kNN:", neighbor_block(matrix_train[rows], y_idx[rows], matrix_train[:3]).shape)

Jalankan kNN per lipatan. Test memakai rata-rata hasil kelima lipatan (sebarannya sama dengan
OOF, karena tiap kali dicari di 80% train).

In [ ]:
def embed(texts):
    encoder = SentenceTransformer(EMBEDDING_MODEL, device="cpu")
    texts = list(TEXT_PREFIX + texts)
    return encoder.encode(texts, batch_size=64, normalize_embeddings=True)


space_train = matrix_train
space_test = matrix_test
# space_train = embed(full_train)   # alt: vektor kalimat (beri # dua baris di atas)
# space_test = embed(full_test)
n_neighbor_cols = 2 + 3 * len(CLASSES)
neighbor_train = np.zeros((len(train), n_neighbor_cols), dtype="float32")
neighbor_test = np.zeros((len(test), n_neighbor_cols), dtype="float32")
START = time.time()
for idx_fit, idx_val in folds:
    ref = space_train[idx_fit]
    neighbor_train[idx_val] = neighbor_block(ref, y_idx[idx_fit], space_train[idx_val])
    neighbor_test += neighbor_block(ref, y_idx[idx_fit], space_test) / N_FOLDS
neighbor_names = ["tetangga_sim1", "tetangga_sim_rata"]
for label in CLASSES:
    neighbor_names = neighbor_names + [f"tetangga_porsi_{label}", f"tetangga_maks_{label}"]
for label in CLASSES:
    neighbor_names.append(f"pusat_{label}")
vote = neighbor_train[:, 2:2 + 2 * len(CLASSES):2].argmax(axis=1)
print("kNN saja (suara tetangga) F1 makro OOF:", round(f1_score(y_idx, vote, average="macro"), 4),
      "| detik", round(time.time() - START))
twins = int((neighbor_train[:, 0] >= 0.95).sum())
print("baris train dengan tetangga kembar (sim >= 0.95):", twins)

## Fitur 3: statistik teks
Sama dengan `teks_tabular`: panjang, kata, huruf besar, angka, tanda baca per kolom teks.

In [ ]:
def text_stats(table, col):
    text = table[col].fillna("")
    words = text.str.split()
    out = pd.DataFrame(index=table.index)
    out[f"{col}_n_char"] = text.str.len()
    out[f"{col}_n_word"] = words.str.len()
    out[f"{col}_n_upper"] = text.str.count(r"[A-Z]")
    out[f"{col}_n_digit"] = text.str.count(r"\d")
    out[f"{col}_n_punct"] = text.str.count(r"[^\w\s]")
    out[f"{col}_n_excl"] = text.str.count("!")
    out[f"{col}_upper_ratio"] = out[f"{col}_n_upper"] / (out[f"{col}_n_char"] + 1)
    return out


all_data = pd.concat([train, test], ignore_index=True)
stat_blocks = []
for col in TEXT_COLS:
    stat_blocks.append(text_stats(all_data, col))
stats = pd.concat(stat_blocks, axis=1)
print("statistik teks:", stats.shape)

## Rakit fitur
Kolom tabel + statistik + OOF model teks + tetangga. `category` dibangun dari train + test.
`GROUPS` mencatat kolom tiap kelompok untuk ablasi dan grafik sumbangan.

In [ ]:
level1_all = np.vstack([np.hstack(level1_train), np.hstack(level1_test)])
neighbor_all = np.vstack([neighbor_train, neighbor_test])
blocks = [all_data[NUM_COLS], stats,
          pd.DataFrame(level1_all, columns=level1_names),
          pd.DataFrame(neighbor_all, columns=neighbor_names)]
for col in CAT_COLS:
    blocks.append(all_data[col].astype("category"))
features = pd.concat(blocks, axis=1)
X_train = features.iloc[: len(train)].reset_index(drop=True)
X_test = features.iloc[len(train):].reset_index(drop=True)
GROUPS = {"tabel": list(NUM_COLS) + list(CAT_COLS), "statistik": list(stats.columns),
          "oof_teks": level1_names, "tetangga": neighbor_names}
print("fitur:", X_train.shape)
for group in GROUPS:
    print("  ", group, len(GROUPS[group]), "kolom")

## Model + CV (lipatan yang sama)
LightGBM dangkal + `colsample_bytree` kecil: fitur OOF sangat kuat, jadi pohon mudah hanya
memakai itu. Alternatif ber-`#`: kelas timpang.

In [ ]:
model = LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=15, min_child_samples=40,
                       colsample_bytree=0.4, subsample=0.8, subsample_freq=1, reg_lambda=1.0,
                       random_state=SEED, n_jobs=-1, verbose=-1)
# model.set_params(class_weight="balanced")   # alt: kelas timpang berat


def run_cv(model, X, X_test, verbose=True):
    oof = np.zeros((len(X), len(CLASSES)), dtype="float32")
    test_pred = np.zeros((len(X_test), len(CLASSES)), dtype="float32")
    for fold, (idx_fit, idx_val) in enumerate(folds):
        fitted = copy.deepcopy(model)
        fitted.fit(X.iloc[idx_fit], y_idx[idx_fit])
        oof[idx_val] = fitted.predict_proba(X.iloc[idx_val])
        test_pred += fitted.predict_proba(X_test) / N_FOLDS
        score = f1_score(y_idx[idx_val], oof[idx_val].argmax(axis=1), average="macro")
        if verbose:
            print("lipatan", fold, "F1 makro", round(score, 4))
    return oof, test_pred


START = time.time()
oof, test_pred = run_cv(model, X_train, X_test)
SECONDS = round(time.time() - START)

## Evaluasi OOF
Metrik lomba belum tentu F1 makro: ganti `SCORE` bila perlu.

In [ ]:
oof_labels = oof.argmax(axis=1)
SCORE = f1_score(y_idx, oof_labels, average="macro")
print("F1 makro OOF:", round(SCORE, 4), "| akurasi:", round(accuracy_score(y_idx, oof_labels), 4),
      "| log loss:", round(log_loss(y_idx, oof), 4), "| detik:", SECONDS)
print(classification_report(y_idx, oof_labels, target_names=CLASSES.astype(str)))

## Ablasi: sumbangan tiap kelompok fitur
Model sama (lebih cepat), kelompok ditambah satu per satu. Disimpan ke `ablasi.csv` untuk slide.

In [ ]:
ablation_model = copy.deepcopy(model).set_params(n_estimators=250, learning_rate=0.06)
columns = []
rows = []
for group in ["tabel", "statistik", "oof_teks", "tetangga"]:
    columns = columns + GROUPS[group]
    if len(columns) == 0:
        continue
    ablation_oof, _ = run_cv(ablation_model, X_train[columns], X_test[columns], False)
    score = f1_score(y_idx, ablation_oof.argmax(axis=1), average="macro")
    rows.append({"fitur sampai": group, "kolom": len(columns), "F1 makro OOF": round(score, 4)})
ablation = pd.DataFrame(rows)
ablation.to_csv(f"{OUTPUT_FOLDER}/ablasi.csv", index=False)
print(ablation.to_string(index=False))

## Pentingnya kelompok fitur (bahan slide)
Satu model di seluruh train, `feature_importances_` dijumlah per kelompok. PNG + CSV disimpan
di `OUTPUT_FOLDER` (saat presentasi data tidak bisa dibuka lagi).

In [ ]:
full_model = copy.deepcopy(model).fit(X_train, y_idx)
importance = pd.Series(full_model.feature_importances_, index=X_train.columns)
by_group = {}
for group in GROUPS:
    by_group[group] = importance.reindex(GROUPS[group]).sum()
by_group = pd.Series(by_group).sort_values()
share = by_group / by_group.sum()
share.rename("porsi").to_csv(f"{OUTPUT_FOLDER}/kelompok_fitur.csv")
top = importance.sort_values(ascending=False).head(15)
print(top.round(1).to_string())
figure, ax = plt.subplots(figsize=(9, 3.5))
ax.barh(share.index, share.values, color="#0072B2")
ax.set_xlabel("porsi pemakaian di pohon (split)")
figure.savefig(f"{OUTPUT_FOLDER}/kelompok_fitur.png", dpi=200, bbox_inches="tight")
plt.show()

## Simpan OOF
Format bersama (dibaca `ensemble_oof`): `oof_<MODEL_NAME>.npy` dan `test_<MODEL_NAME>.npy`.
Catatan: OOF ini sudah memakai OOF model teks, jadi di `ensemble_oof` ia berkorelasi tinggi
dengan `tfidf_logreg`; blending tetap sah karena lipatannya sama.

In [ ]:
np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", oof.astype("float32"))
np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", test_pred.astype("float32"))

log_path = f"{OOF_FOLDER}/percobaan.csv"
note = f"{X_train.shape[1]} fitur: oof {list(LEVEL1) + EXTRA_OOF} + knn{N_NEIGHBORS}"
log_row = pd.DataFrame([{"waktu": time.strftime("%Y-%m-%d %H:%M:%S"),
                         "notebook": "stacking_teks_tabular", "model": MODEL_NAME,
                         "metrik": "f1_makro", "skor_cv": round(SCORE, 4), "detik": SECONDS,
                         "catatan": note}])
log_row.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
history = pd.read_csv(log_path)
print(history.tail(3).to_string(index=False))

## Prediksi test + submission
Kelas = `argmax` rata-rata peluang model per lipatan. Id dan urutan dari sample_submission.

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
pred_labels = CLASSES[test_pred.argmax(axis=1)]
pred_by_id = pd.Series(pred_labels, index=test[ID_COL].values)

submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUB_COL] = sample[ID_COL].map(pred_by_id).values
submission.to_csv(f"{OUTPUT_FOLDER}/submission.csv", index=False)
print(submission.shape)
print(submission[SUB_COL].value_counts().sort_index())

## Cek submission
Hanya mencetak. Hasil akhir harus `MASALAH: 0`.

In [ ]:
written = pd.read_csv(f"{OUTPUT_FOLDER}/submission.csv")
problems = []
if list(written.columns) != list(sample.columns):
    problems.append(f"kolom/urutan beda: {list(written.columns)} vs {list(sample.columns)}")
if len(written) != len(sample) or list(written[ID_COL]) != list(sample[ID_COL]):
    problems.append("jumlah/urutan id beda dari sample")
if written[SUB_COL].isna().sum() > 0:
    problems.append(f"{SUB_COL}: {written[SUB_COL].isna().sum()} kosong")
if written[SUB_COL].dtype != sample[SUB_COL].dtype:
    problems.append(f"{SUB_COL}: tipe {written[SUB_COL].dtype}, sample {sample[SUB_COL].dtype}")
foreign = set(written[SUB_COL].dropna()) - set(CLASSES.tolist())
if len(foreign) > 0:
    problems.append(f"{SUB_COL}: label di luar train {sorted(foreign)[:5]}")
for problem in problems:
    print("-", problem)
print("MASALAH:", len(problems))

## Cek dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].values, index=key[ID_COL].values)
truth = submission[ID_COL].map(key_by_id)
test_f1 = f1_score(truth, submission[SUB_COL], average="macro")
print("F1 makro test:", round(test_f1, 4), "| F1 makro OOF:", round(SCORE, 4),
      "| selisih:", round(SCORE - test_f1, 4))
print("akurasi test :", round(accuracy_score(truth, submission[SUB_COL]), 4))